# Factorial DOE: oil-company experiment

Industrial designed experiment from an oil company. Four blend
components are varied across 19 runs to improve volumetric heat
capacity. Three of the components (A, B, D) are continuous and were
varied in coded units; the fourth (C) is a binary factor: present or
absent. The aim is to identify which factors and interactions actually
drive the response so that future experimentation can focus there.

**Data.** `oil-company-doe.csv` from
[openmv.net](https://openmv.net/info/oil-company-doe). 19 rows, five
columns: `A`, `B`, `C`, `D`, `y`.

**What we do.** Run the package's `analyze_experiment` with an
interactions model, read the coefficient table and ANOVA, build a
Pareto plot of standardised effects, and check the residual
diagnostics.

**Adapted from** the *Design and analysis of experiments* chapter of
the [Process Improvement using Data](https://learnche.org/pid) book
(CC BY-SA 4.0).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from process_improve.experiments.analysis import analyze_experiment

## Load and inspect

In [ ]:
url = "https://openmv.net/file/oil-company-doe.csv"
doe = pd.read_csv(url, index_col=0)
if doe.shape[1] < 5:
    doe = pd.read_csv(url)
doe.columns = [c.strip() for c in doe.columns]
print(f"Shape: {doe.shape}")
print(f"Columns: {list(doe.columns)}")
doe.head()

In [ ]:
doe.describe(include="all")

## Fit the interactions model

`analyze_experiment` accepts a design DataFrame and the name of the
response column, builds a formula, fits it via statsmodels, and
returns a dictionary keyed by the requested analysis types. With
`model="interactions"` we ask for main effects plus all two-factor
interactions; that is 4 + 6 = 10 terms, which fits comfortably in 19
runs.

In [ ]:
result = analyze_experiment(
    doe,
    response_column="y",
    model="interactions",
    analysis_type=[
        "coefficients",
        "anova",
        "effects",
        "significance",
        "residual_diagnostics",
    ],
)
ms = result["model_summary"]
print(f"formula     : {ms['formula']}")
print(f"R2          : {ms['r_squared']:.3f}")
print(f"R2 (adj)    : {ms['r_squared_adj']:.3f}")
print(f"R2 (pred)   : {ms['r_squared_pred']:.3f}")
print(f"obs / df_resid: {ms['n_obs']} / {ms['df_residual']}")

## Coefficient table

Sort by absolute t-value to put the most impactful terms at the top.
Coefficients with a small p-value are the ones that the data picks out
as important; coefficients whose confidence interval straddles zero
are not.

In [ ]:
coef_df = pd.DataFrame(result["coefficients"]).set_index("term")
coef_df = coef_df.reindex(coef_df["t_value"].abs().sort_values(ascending=False).index)
coef_df.round(3)

## ANOVA

In [ ]:
anova = pd.DataFrame(result["anova_table"]).set_index("source")
anova.round(3)

## Pareto plot of standardised effects

Dividing each effect estimate by its standard error gives a unitless
comparison of how strong each term is relative to the noise. Plotting
the absolute values as a horizontal bar chart from largest to smallest
is the classical Pareto plot.

In [ ]:
effects = pd.Series(result["effects"], dtype=float)
errors = pd.Series(result["effect_std_errors"], dtype=float).reindex(effects.index)
standardised = (effects / errors).abs().sort_values()

_fig, ax = plt.subplots(figsize=(7, 0.35 * max(4, len(standardised)) + 1.0))
ax.barh(standardised.index, standardised.values, color="#1f77b4")
ax.axvline(2, color="r", linestyle="--", linewidth=1, label="|t| = 2 (rough significance)")
ax.set_xlabel("|effect / standard error|")
ax.set_title("Pareto of standardised effects")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

## Residual diagnostics

In [ ]:
diag = result["residual_diagnostics"]
fitted = np.asarray(diag["fitted_values"], dtype=float)
resid = np.asarray(diag["residuals"], dtype=float)

_fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].scatter(fitted, resid, s=30, color="#1f77b4")
axes[0].axhline(0, color="k", linewidth=0.7)
axes[0].set_xlabel("Fitted")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs fitted")

axes[1].hist(resid, bins=8, color="#1f77b4", edgecolor="k")
axes[1].set_xlabel("Residual")
axes[1].set_title("Residual distribution")
plt.tight_layout()
plt.show()

print(f"Shapiro-Wilk p-value : {diag['shapiro_wilk']['p_value']:.3f}")
print(f"Durbin-Watson        : {diag['durbin_watson']:.2f}")
print(f"Breusch-Pagan p-value: {diag['breusch_pagan']['p_value']:.3f}")

## What to try next

- Drop the terms below the `|t| = 2` line and re-fit. The model R
  squared will go down a touch but the adjusted R squared usually goes
  up because the noise terms penalise the unadjusted statistic.
- Pass `analysis_type="lack_of_fit"` to ask whether the residual
  variation is consistent with pure replication error, which tells you
  whether a curvature term is missing.
- For follow-up experimentation, the
  `recommend_strategy()`
  function will propose where to run the next batch of experiments
  given the coefficients you have already estimated.